# 2/3 — Data Preprocessing (bản chạy được trên Linux, 2 file pcap)

Copy của `Data_preprocessing_CIC-IoT2023.ipynb`. Chạy SAU phần A của `1_GNN4ID_pcap.ipynb`.

Luồng: `split_csv` (lọc theo MAC của attacker + tách train/test) → `Combining_classes`
(gộp sub-class thành class lớn, gán cột `Label`) → gộp thành 1 file train + 1 file test.

In [ ]:
from Utility.Functions import *
import pandas as pd
import glob
import os
from tqdm import tqdm

In [ ]:
# [linux] Toàn bộ path của notebook gốc là Windows (F:/CIC_IOT/...) -> đổi sang path máy này.
# 2 file pcap gốc KHÔNG bị đụng tới: mọi thứ sinh ra nằm trong .../Debug and Trace/nb_run/
import os

BASE          = "/home/tutay/Tutay/Tutay_Sec/XG_NID"
PCAP_SRC      = os.path.join(BASE, "data", "Debug and Trace")              # 2 file pcap input
WORK          = os.path.join(PCAP_SRC, "nb_run")                           # thư mục làm việc
Out_Directory = os.path.join(WORK, "Packet_Level_Data")                    # pcap sau khi đổi tên
Out_path      = os.path.join(WORK, "Extracted_Flow_Features") + os.sep     # csv + graph objects
print("WORK     =", WORK)
print("Out_path =", Out_path)

### Tách Train / Test

`split_csv` giữ lại flow có MAC attacker (với class tấn công), loại flow chạm MAC attacker
(với Benign), rồi tách 80/20. Phần train ghi đè lên chính file csv, phần test ghi vào
`Extracted_Flow_Features/test/`.

In [ ]:
directory = Out_path
List_of_CSV_File = glob.glob(os.path.join(directory, 'features', '*csv'))   # rolling features đã tính trên toàn file
print(List_of_CSV_File)
# Lọc MAC attacker + chia theo THỜI GIAN 80/20 + cap per file; ghi ra split/train và split/test, không ghi đè
for files in tqdm(List_of_CSV_File):
    split_csv(files, out_dir=os.path.join(directory, 'split'))


### Gộp sub-class thành class lớn

[linux] Chỉ để 2 class có trong 2 file pcap; để cả 8 class thì `pd.concat([])` sẽ lỗi vì
không có file cho các class còn lại.

In [ ]:
Attack_Classes = ['BruteForce', 'WebBased']   # 2 pcap -> chỉ có 2 class; None = mọi class tìm thấy
label_dict = {'Benign': 0,'WebBased': 1,'Spoofing': 2,'Recon': 3,'Mirai': 4,'Dos': 5,'DDos': 6,'BruteForce': 7}
# [guard] báo lỗi rõ nếu chưa chạy split_csv
for cls in Attack_Classes:
    assert glob.glob(os.path.join(directory, 'split', 'train', cls + '*')), (
        "Không có file split/train/%s* trong %s -> chạy cell split_csv trước" % (cls, directory))
## Gộp sub-attack của mỗi class: dedup, cap 20k/4k theo tỷ lệ sub-attack, loại test trùng train (toàn cục),
## rồi oversample CHỈ train lên 20k (paper Table 4); class_weights.json cũng được ghi ra.
report = Combining_classes(os.path.join(directory, 'split'), Attack_Classes, label_dict=label_dict,
                           oversample=True, out_dir=os.path.join(directory, 'combined'))
report


### Gộp thành df_class_8_train.csv / df_class_8_test.csv


In [ ]:
## 1 file train + 1 file test: concat các class, drop 29 cột định danh (cell 9-13 cũ) và assert header 97 cột
result = build_class8_csvs(os.path.join(directory, 'combined'), directory)
result


Xong → quay lại **phần B** của `1_GNN4ID_pcap.ipynb` để sinh graph objects.